# CSD355 – Real data preparation (run in Google Colab)

The raw datasets are many GB, so they are downloaded and reduced **here** instead of on a laptop.
The output is a few small CSVs in the common `(entity_id, timestamp, demand_mbps)` format.
Copy those into `data/processed/` of the project and set `source: real` in `config.yaml`.

All parsing logic lives in the project (`scripts/prepare_real_data.py`, `src/bwalloc/data/loaders.py`),
so this notebook only orchestrates. Runtime: CPU is enough.

## 1. Get the project code
Upload a zip of the project folder (without `.venv/`, `data/raw/`, `results/`), **or** put the folder in Google Drive and mount it.

In [ ]:
# Option A: upload a zip of the project
from google.colab import files
uploaded = files.upload()          # choose FDS.zip
!unzip -q -o FDS.zip -d /content/
%cd /content/FDS

In [ ]:
# Option B: project folder in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/FDS

In [ ]:
!pip install -q -e .

## 2. Milan – Telecom Italia Big Data Challenge (ODbL)
Source: Harvard Dataverse `doi:10.7910/DVN/EGZHFV`, 62 daily files of ~330 MB.
The dataset has a Dataverse *guestbook* (email required); the script submits it per file.
Files are streamed one at a time; the 150 most active cells of the first day are kept for all days.
28 days ≈ 9 GB of downloads, about 15–30 minutes in Colab. The output CSV is checkpointed after every day.

In [ ]:
import os
# The Milan dataset has a guestbook: an email is required (name optional).
os.environ['MILAN_GUESTBOOK_EMAIL'] = input('Guestbook email: ')
os.environ['MILAN_GUESTBOOK_NAME'] = input('Name (optional): ')
!python -u scripts/prepare_real_data.py milan --days 28 --cells 150 --start 2013-11-04

## 3. MAWI backbone traces (optional, raw pcap route)
**Already done without pcaps:** `scripts/fetch_mawi_agurim.py` pulls MAWI's continuous agurim aggregate flows (used in the project). This cell is only for the raw-pcap route; one 15-minute capture is ~24 GB.
MAWI publishes 15-minute daily captures and longer *Day-in-the-Life* (DITL) captures at
<https://mawi.wide.ad.jp/mawi/>. Only DITL captures (48–72 h) are long enough for a time series;
use `--window 1min` and note that `lag_7d` / `lag_24h` will be dropped automatically.
Fill in the trace URL(s) you choose:

In [ ]:
!apt-get -qq install -y tshark > /dev/null
TRACE_URLS = [
    # "https://mawi.wide.ad.jp/mawi/ditl/ditl2023/<file>.pcap.gz",
]
import os, subprocess
csvs = []
for i, url in enumerate(TRACE_URLS):
    !wget -q -O trace.pcap.gz "$url" && gunzip -f trace.pcap.gz
    out = f"mawi_{i}.csv"
    subprocess.run(f"tshark -r trace.pcap -T fields -E separator=, -e frame.time_epoch -e ip.src -e frame.len > {out}", shell=True, check=True)
    os.remove("trace.pcap"); csvs.append(out)
if csvs:
    !python scripts/prepare_real_data.py packets --name mawi --inputs {" ".join(csvs)} --window 1min

## 4. CAIDA anonymized traces (only if access is granted)
CAIDA requires a request and acceptance of the CAIDA Terms of Use. Once you have a pcap,
export it with tshark exactly as above and run:
`python scripts/prepare_real_data.py packets --name caida --inputs caida.csv --window 1min`

## 5. FCC Measuring Broadband America
Raw data releases: <https://www.fcc.gov/reports-research/reports/measuring-broadband-america/raw-data-measuring-broadband-america-2023>
(pick a month). Use the data-usage table; check its column names first and pass them if they differ.

In [ ]:
FCC_URL = ""   # paste the monthly raw-data archive URL here
if FCC_URL:
    !wget -q -O fcc.tar.gz "$FCC_URL" && mkdir -p fcc && tar -xzf fcc.tar.gz -C fcc && rm fcc.tar.gz
    !find fcc -iname "*usage*"
    # !head -3 fcc/<path>/curr_datausage.csv
    # !python scripts/prepare_real_data.py fcc --inputs fcc/<path>/curr_datausage.csv --unit-col unit_id --time-col dtime --bytes-cols wan_rx_bytes wan_tx_bytes

## 6. Check and download the processed files

In [ ]:
import pandas as pd, glob
for p in sorted(glob.glob("data/processed/*.csv")):
    df = pd.read_csv(p, parse_dates=["timestamp"])
    print(p, df.shape, df.entity_id.nunique(), "users", df.timestamp.min(), "->", df.timestamp.max())
!cd data/processed && zip -q ../../processed.zip *.csv provenance.json
files.download("processed.zip")